In [6]:
import pandas as pd
import numpy as np
import xarray as xr
import cmocean
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.ticker import LogLocator, FormatStrFormatter
from matplotlib.colors import LogNorm
import matplotlib.path as mpath
import gsw
import earthaccess
import contourpy
from tqdm import tqdm
from importlib import reload
import SO_tools as tools; reload(tools)

<module 'SO_tools' from '/Users/lilah/Documents/IBIS_Project/SO_tools.py'>

# Identify Southern Ocean Fronts Using RG09

In [11]:
# ds_RG = xr.open_dataset('/raid/walkersl/GOBAI_O2_analysis/Data/variables/GOBAI-O2-v2.3_conservative_temperature.nc')
ds_RG = xr.open_dataset('/Users/lilah/Documents/IBIS_Project/Data_Summer/GOBAI-O2-v2.3_conservative_temperature.nc')

# convert lon to 0-360 (GOBAI is weird and has a 20 - 380 lon coordinates)
ds_RG = tools.lon_convert(ds_RG)

def build_front_fields(ds_temp):
    """Compute continuous fields, zone masks, and front_masks for all timesteps."""
    temp = ds_temp['temp'].sel(lat=slice(-65,-30))
    lat = ds_temp['lat'].sel(lat=slice(-65,-30))

    theta_100 = temp.sel(pres=100, method='nearest').drop_vars('pres')
    theta_400 = temp.sel(pres=400, method='nearest').drop_vars('pres')

    temp_u200 = temp.sel(pres=slice(2.5,200))
    valid = ~temp_u200.isnull().all(dim='pres')
    temp_for_min = temp_u200.where(valid, other=np.inf)
    imin = temp_for_min.argmin(dim='pres')
    theta_min = temp_for_min.isel(pres=imin).where(valid).drop_vars('pres')

    # front masks (thresholded points, used only to find each front's latitude per lon/time)
    STF = xr.where(abs(theta_100 - 11.0) <= 0.1, 1, np.nan)
    SAF = xr.where(abs(theta_400 - 5.0) <= 0.1, 1, np.nan)
    PF  = xr.where(abs(theta_min - 2.0) <= 0.1, 1, np.nan)

    STF_lat = STF.fillna(0).astype(bool) * lat
    SAF_lat = SAF.fillna(0).astype(bool) * lat
    PF_lat  = PF.fillna(0).astype(bool) * lat
    STF_latline = STF_lat.max(dim='lat')
    SAF_latline = SAF_lat.max(dim='lat')
    PF_latline  = PF_lat.max(dim='lat')

    lat2d = lat.broadcast_like(theta_100)
    STZ = xr.where(lat2d > STF_latline, 1, np.nan).rename('STZ_mask')
    SAZ = xr.where((lat2d <= STF_latline) & (lat2d > SAF_latline), 1, np.nan).rename('SAZ_mask')
    PAZ = xr.where((lat2d <= SAF_latline) & (lat2d > PF_latline), 1, np.nan).rename('PAZ_mask')

    ds_out = xr.Dataset({
        'theta_100': theta_100.rename('theta_100'),
        'theta_400': theta_400.rename('theta_400'),
        'theta_min': theta_min.rename('theta_min'),
        'STZ_mask': STZ,
        'SAZ_mask': SAZ,
        'PAZ_mask': PAZ,
    })
    ds_out.attrs['description'] = (
        'Continuous fields for front contouring (STF=11.0C@100db, SAF=5.0C@400db, '
        'PF=2.0C theta_min upper 200db) + derived zone masks (STZ/SAZ/PAZ)'
    )
    return ds_out

front_ds = build_front_fields(ds_RG)
# front_ds.to_netcdf('/raid/walkersl/SO_Argo/Data/SO_front_fields_zones_monthly.nc')                            

ValueError: found the following matches with the input file in xarray's IO backends: ['netcdf4', 'h5netcdf']. But their dependencies may not be installed, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

In [12]:
# front_ds = xr.open_dataset('/raid/walkersl/SO_Argo/Data/SO_front_fields_zones_monthly.nc')
front_ds = xr.open_dataset('/Users/lilah/Documents/IBIS_Project/Data_Summer/SO_front_fields_zones_monthly.nc')
                          
front_field_levels = {
    'STF': ('theta_100', 11.0),
    'SAF': ('theta_400', 5.0),
    'PF':  ('theta_min', 2.0),
}

def get_front_contours(front_ds, timestep, front_field_levels):
    """Return dict of {front_name: list of (N,2) lon/lat vertex arrays} for one timestep."""
    contours = {}
    for front, (varname, level) in front_field_levels.items():
        field = front_ds[varname].sel(time=timestep, method='nearest')
        gen = contourpy.contour_generator(x=field['lon'].values, y=field['lat'].values, z=field.values)
        contours[front] = gen.lines(level)
    return contours

# timestep = '2014-06-15'
# front_contours = get_front_contours(front_ds, timestep, front_field_levels)

ValueError: found the following matches with the input file in xarray's IO backends: ['netcdf4', 'h5netcdf']. But their dependencies may not be installed, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

In [43]:
temp = ds_RG['temp'].sel(lat=slice(-65,-30),pres=2.5)
lon = ds_RG['lon']
lat = ds_RG['lat'].sel(lat=slice(-65,-30))
time = ds_RG['time']

for i in tqdm(range(len(time))):

    timestep = time.isel(time=i).values
    timestep = pd.to_datetime(timestep, utc=True)
    time_label = timestep.strftime("%Y-%m-%d")
    
    front_contours = get_front_contours(front_ds, time_label, front_field_levels)

    fig = plt.figure(figsize=(6,6))
    crs0 = ccrs.PlateCarree()
    crs = ccrs.SouthPolarStereo()
    theta = np.linspace(0, 2*np.pi, 100)
    map_circle = mpath.Path(np.vstack([np.sin(theta), np.cos(theta)]).T * 0.5 + [0.5, 0.5])
    cmap = cmocean.cm.thermal
    tick_size = 12
    colorbar_label = 'Temp (C)'
    axis_size= 14

    ax = fig.add_subplot(111, projection=ccrs.SouthPolarStereo())
    ax.set_extent([-180, 180, -90, -30], crs=ccrs.PlateCarree())
    ax.set_boundary(map_circle, transform=ax.transAxes)

    plot = temp.sel(time=time_label).plot(x='lon', y='lat', vmin=-2, vmax=25, cmap=cmap, transform=crs0, add_colorbar=False)

    front_colors = {'STF': 'green', 'SAF': 'fuchsia', 'PF': 'cyan'}
    for front, lines in front_contours.items():
        for verts in lines:
            ax.plot(verts[:,0], verts[:,1], color=front_colors[front],
                     transform=ccrs.PlateCarree(), linewidth=1.5)

    coast_50m = cfeature.NaturalEarthFeature('physical', 'land', '110m', edgecolor='k', facecolor='darkgray')
    ax.add_feature(coast_50m)
    ax.set_boundary(map_circle, transform=ax.transAxes)

    cbar = fig.colorbar(plot, fraction=0.012, pad=0.01)
    cbar.ax.tick_params(labelsize=tick_size)
    cbar.set_label(colorbar_label, size=axis_size)

    gl = ax.gridlines(crs=crs0, draw_labels=True, x_inline=False, y_inline=False,
                      linewidth=1, color='grey', alpha=0.3, linestyle='-')
    gl.xlabel_style = {'size': tick_size, 'color': 'k'}
    gl.ylabel_style = {'size': tick_size, 'color': 'k'}
    gl.top_labels = False

    #study region box
    lon_lim=[150,180]
    lat_lim=[-68,-55]
    rect = mpatches.Rectangle(
        (lon_lim[0], lat_lim[0]),
        lon_lim[1] - lon_lim[0],
        lat_lim[1] - lat_lim[0],
        transform=ccrs.PlateCarree(),
        facecolor='none', edgecolor='red', linewidth=1.5, linestyle='-'
    )
    ax.add_patch(rect)

    ax.set_title(f'Temperature at Surface with Fronts— {timestep}', size=16, weight='bold')

    fig.savefig(f'/raid/walkersl/SO_Argo/Figures/maps/RG09_fronts/RG09_SST_with_SO_fronts_{time_label}.png',
                    bbox_inches='tight')
    plt.close()



100%|██████████| 252/252 [02:06<00:00,  1.99it/s]


In [ ]:
tools.create_gif('/raid/walkersl/SO_Argo/Figures/maps/RG09_fronts/','/raid/walkersl/SO_Argo/Figures/maps/RGO9_SST_with_SO_fronts_animation.gif')

MovieWriter imagemagick unavailable; using Pillow instead.
